In [1]:
import os
import copy
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# Constants / path
data_dir = "image-results-spoofing/data"
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")
batch_size = 32
learning_rate = 0.001
num_epochs = 200
RANDOM_STATE = 1082
criterion = nn.CrossEntropyLoss()

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
# Data transforms
train_transforms = transforms.Compose([
    transforms.Resize((64, 82)), # Resize to match the input size of photo
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
test_transforms = transforms.Compose([
    transforms.Resize((64, 82)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [5]:
# Datasets and DataLoaders
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

dataloaders = {'train': train_loader, 'test': test_loader}
classes = train_dataset.classes

In [6]:
# CNN Model Definition
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.layer1 = self.ConvModule(in_features=3, out_features=64)  # 16x16
        self.layer2 = self.ConvModule(in_features=64, out_features=128)  # 8x8
        self.layer3 = self.ConvModule(in_features=128, out_features=256)  # 4x4
        self.layer4 = self.ConvModule(in_features=256, out_features=512)  # 2x2
        self.gap = nn.AdaptiveAvgPool2d((2, 2))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * 2 * 512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

    def ConvModule(self, in_features, out_features):
        return nn.Sequential(
            nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_features),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )


model = CNN(num_classes=len(classes)).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [7]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs, device):
    since = time.time()
    history = {'train_loss': [], 'train_accuracy': [], 'test_loss': [], 'test_accuracy': []}

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += preds.eq(labels).sum().item()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects / len(dataloaders[phase].dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2%}')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_accuracy'].append(epoch_acc)

            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best test Acc: {best_acc:.2%}')

    model.load_state_dict(best_model_wts)
    return model, history

# Train
model, history = train_model(model, dataloaders, criterion, optimizer, num_epochs, device)

Epoch 1/200
Train Loss: 0.7682, Acc: 77.89%
Test Loss: 0.7589, Acc: 77.92%

Epoch 2/200
Train Loss: 0.7545, Acc: 77.92%
Test Loss: 0.7312, Acc: 77.92%

Epoch 3/200
Train Loss: 0.7186, Acc: 78.13%
Test Loss: 0.8357, Acc: 74.37%

Epoch 4/200
Train Loss: 0.6561, Acc: 80.17%
Test Loss: 1.0148, Acc: 57.61%

Epoch 5/200
Train Loss: 0.6030, Acc: 82.15%
Test Loss: 0.5715, Acc: 83.65%

Epoch 6/200
Train Loss: 0.5724, Acc: 83.22%
Test Loss: 0.6468, Acc: 80.39%

Epoch 7/200
Train Loss: 0.5548, Acc: 83.84%
Test Loss: 1.4989, Acc: 43.44%

Epoch 8/200
Train Loss: 0.5433, Acc: 84.20%
Test Loss: 0.9684, Acc: 68.43%

Epoch 9/200
Train Loss: 0.5338, Acc: 84.42%
Test Loss: 0.5581, Acc: 83.76%

Epoch 10/200
Train Loss: 0.5270, Acc: 84.65%
Test Loss: 0.5099, Acc: 85.28%

Epoch 11/200
Train Loss: 0.5213, Acc: 84.76%
Test Loss: 0.4981, Acc: 85.40%

Epoch 12/200
Train Loss: 0.5116, Acc: 84.95%
Test Loss: 0.5089, Acc: 84.67%

Epoch 13/200
Train Loss: 0.5067, Acc: 85.02%
Test Loss: 0.6213, Acc: 79.02%

Epoch 14

KeyboardInterrupt: 

In [ ]:
model_save_path = "best_cnn_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")